In [0]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

TABLE = "workspace.open_meteo.dim_weather_code"

# (code, description, category, severity_rank)
# severity_rank: 0 = benign ... 5 = most severe (own scale, not WMO)
ROWS: list[tuple[int, str, str, int]] = [
    (0,  "Clear sky",                     "clear",        0),
    (1,  "Mainly clear",                  "clear",        0),
    (2,  "Partly cloudy",                 "cloudy",       1),
    (3,  "Overcast",                      "cloudy",       1),
    (45, "Fog",                           "fog",          2),
    (48, "Depositing rime fog",           "fog",          3),
    (51, "Light drizzle",                 "drizzle",      1),
    (53, "Moderate drizzle",              "drizzle",      2),
    (55, "Dense drizzle",                 "drizzle",      3),
    (56, "Light freezing drizzle",        "drizzle",      3),
    (57, "Dense freezing drizzle",        "drizzle",      4),
    (61, "Slight rain",                   "rain",         2),
    (63, "Moderate rain",                 "rain",         3),
    (65, "Heavy rain",                    "rain",         4),
    (66, "Light freezing rain",           "rain",         4),
    (67, "Heavy freezing rain",           "rain",         5),
    (71, "Slight snow fall",              "snow",         2),
    (73, "Moderate snow fall",            "snow",         3),
    (75, "Heavy snow fall",               "snow",         4),
    (77, "Snow grains",                   "snow",         2),
    (80, "Slight rain showers",           "rain",         2),
    (81, "Moderate rain showers",         "rain",         3),
    (82, "Violent rain showers",          "rain",         4),
    (85, "Slight snow showers",           "snow",         3),
    (86, "Heavy snow showers",            "snow",         4),
    (95, "Thunderstorm",                  "thunderstorm", 4),
    (96, "Thunderstorm with slight hail", "thunderstorm", 5),
    (99, "Thunderstorm with heavy hail",  "thunderstorm", 5),
]

schema = StructType([
    StructField("weather_code",   IntegerType(), nullable=False),
    StructField("description",    StringType(),  nullable=False),
    StructField("category",       StringType(),  nullable=False),
    StructField("severity_rank",  IntegerType(), nullable=False),
])

source: DataFrame = spark.createDataFrame(ROWS, schema)

# Create the table once; a no-op on reruns
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        weather_code  INT    NOT NULL,
        description   STRING NOT NULL,
        category      STRING NOT NULL,
        severity_rank INT    NOT NULL
    )
    COMMENT 'WMO weather code lookup as used by Open-Meteo. severity_rank is a 0-5 judgement scale, not a WMO standard.'
""")

(
    DeltaTable.forName(spark, TABLE).alias("t")
    .merge(source.alias("s"), "t.weather_code = s.weather_code")
    .whenMatchedUpdateAll(
        condition="""
            t.description   <> s.description
            OR t.category      <> s.category
            OR t.severity_rank <> s.severity_rank
        """
    )
    .whenNotMatchedInsertAll()
    # .whenNotMatchedBySourceDelete()   # uncomment to drop codes removed from ROWS
    .execute()
)

assert spark.table(TABLE).count() >= len(ROWS), "dim_weather_code is missing rows"